# Experiment 5: CGM + time_since_meal + image (clip, gated)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [3]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+clip (gated)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'clip'

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gated_clip'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_clip_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_clip_val_stability_summary.json'

In [4]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

val_participant_ids = data.load_multimodal_participant_ids('val')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print()
print('number of test participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print()
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]


number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]


In [5]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = MultimodalGRUPredictor(img_feat_dim=768, **{'gru_hidden_size': 256, 'gru_num_layers': 3, 'gru_dropout': 0.2})

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gated_clip' /f'gated_cgm_time_clip_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)

    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    val_results_per_participant = metrics.evaluate_per_participant(
        y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gated_cgm_time_clip_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Validation results — CGM+time+clip (gated) | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.43 | MAE 8.55 | gRMSE 13.83 | Zone A 94.87% | A+B 99.45%
 30 min | RMSE 18.15 | MAE 12.22 | gRMSE 20.82 | Zone A 89.06% | A+B 99.25%
 60 min | RMSE 25.54 | MAE 16.96 | gRMSE 30.52 | Zone A 80.86% | A+B 98.69%
 90 min | RMSE 30.33 | MAE 20.23 | gRMSE 37.10 | Zone A 75.46% | A+B 98.12%
120 min | RMSE 33.21 | MAE 22.56 | gRMSE 40.96 | Zone A 71.34% | A+B 97.72%

--------------------------------------------------------------------------------
Validation results — CGM+time+clip (gated) | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.01 | MAE 7.48 | gRMSE 12.16 | Zone A 96.72% | A+B 99.64%
 30 min | RMSE 17.72 | MAE 11.84 | gRMSE 20.29 | Zone A 89.63% | A+B 99.27%
 60 min | RMSE 26.04 | MAE 17.16 | gRMSE 31.17 | Zone A 79.82% | A+B 98.64%
 90 min | RMSE 30.75 | MAE 20.50 | gRMSE 37.45 | Zone A 74.39% | A+B 98.14%
120 min | RMSE 33.29 | MAE 22.53 | gRMSE 40.80 | Zone A 70.88% | A+B 97.76%

--------------------------------------------------------------------------------
Validation results — CGM+time+clip (gated) | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


...

 15 min | RMSE 11.66 | MAE 7.94 | gRMSE 12.69 | Zone A 95.68% | A+B 99.57%
 30 min | RMSE 17.93 | MAE 11.99 | gRMSE 20.09 | Zone A 88.82% | A+B 99.27%
 60 min | RMSE 26.04 | MAE 17.21 | gRMSE 30.70 | Zone A 80.06% | A+B 98.69%
 90 min | RMSE 30.77 | MAE 20.50 | gRMSE 37.31 | Zone A 74.59% | A+B 98.12%
120 min | RMSE 33.47 | MAE 22.62 | gRMSE 40.99 | Zone A 71.31% | A+B 97.76%

--------------------------------------------------------------------------------
Validation results — CGM+time+clip (gated) | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.57 | MAE 8.52 | gRMSE 13.59 | Zone A 94.67% | A+B 99.51%
 30 min | RMSE 18.23 | MAE 12.18 | gRMSE 20.49 | Zone A 88.66% | A+B 99.24%
 60 min | RMSE 25.72 | MAE 16.95 | gRMSE 30.53 | Zone A 80.73% | A+B 98.70%
 90 min | RMSE 30.50 | MAE 20.21 | gRMSE 37.33 | Zone A 75.44% | A+B 98.09%
120 min | RMSE 33.35 | MAE 22.53 | gRMSE 41.20 | Zone A 71.49% | A+B 97.69%

--------------------------------------------------------------------------------
Validation results — CGM+time+clip (gated) | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.38 | MAE 8.38 | gRMSE 13.56 | Zone A 95.00% | A+B 99.51%
 30 min | RMSE 18.14 | MAE 12.08 | gRMSE 20.55 | Zone A 89.13% | A+B 99.25%
 60 min | RMSE 25.78 | MAE 16.91 | gRMSE 30.66 | Zone A 80.94% | A+B 98.70%
 90 min | RMSE 30.66 | MAE 20.23 | gRMSE 37.35 | Zone A 75.50% | A+B 98.07%
120 min | RMSE 33.41 | MAE 22.59 | gRMSE 41.01 | Zone A 71.30% | A+B 97.69%

--------------------------------------------------------------------------------


In [7]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [8]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [9]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== CGM+time+clip (gated) validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 12.23 +/- 0.50 (2.49) mg/dL   MAE = 8.34 +/- 0.36 (1.72)   gRMSE = 13.38 +/- 0.55 (3.43)   Zone A = 95.14 +/- 0.63 (4.65)%   Zone A+B = 99.51 +/- 0.06 (0.91)%
 30-min:  RMSE = 18.10 +/- 0.17 (4.02) mg/dL   MAE = 12.13 +/- 0.14 (2.62)   gRMSE = 20.49 +/- 0.27 (6.04)   Zone A = 88.92 +/- 0.38 (5.70)%   Zone A+B = 99.26 +/- 0.01 (1.10)%
 60-min:  RMSE = 25.75 +/- 0.18 (7.84) mg/dL   MAE = 16.97 +/- 0.16 (5.09)   gRMSE = 30.62 +/- 0.25 (11.98)   Zone A = 80.65 +/- 0.48 (6.22)%   Zone A+B = 98.69 +/- 0.02 (1.40)%
 90-min:  RMSE = 30.56 +/- 0.14 (11.03) mg/dL   MAE = 20.26 +/- 0.22 (7.09)   gRMSE = 37.27 +/- 0.16 (16.68)   Zone A = 75.26 +/- 0.58 (8.32)%   Zone A+B = 98.11 +/- 0.03 (2.06)%
120-min:  RMSE = 33.33 +/- 0.09 (12.52) mg/dL   MAE = 22.56 +/- 0.11 (8.06)   gRMSE = 40.98 +/- 0.20 (19.06)   Zone A = 71.30 +/- 0.39 (9.07)%   Zone A+B = 97.73 +/- 0.03 (2.61)%


In [10]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_clip/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_clip/gated_cgm_time_clip_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_clip/gated_cgm_time_clip_stability_summary.json
